# TopicGPT: Temporal Topic Analysis

Analyze topic evolution over time (2000–2025) using **existing best-tuned TopicGPT assignments**.
No retraining needed.

Approach inspired by BERTopic's `topics_over_time()`:
1. Use the tuned model's topic assignments from `tuning_best.pkl`
2. Group documents by year using `submitted_date`
3. Compute per-year topic prevalence, coherence, and word evolution

**Advantage over sliced modeling:** Topics are consistent across all years
(same topic IDs), no alignment needed.

In [1]:
import gc
import time
import pickle
import pandas as pd
import numpy as np
from pathlib import Path
from itertools import combinations
from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer
from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel
import warnings
import ast

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

## Configuration

In [2]:
LIST_SUBJECT = ["cs", "math", "physics"]

BASE_DIR = Path("../../../../data/preprocess")
CHECKPOINT_DIR = Path("../../../../models/topicGpt")
RESULT_DIR = Path("../../../../results/topicGpt/temporal")
VERSION = "v1"

# IRBO config
TOP_N_WORDS = 10
RBO_P = 0.9

# Best tuning config per subject (from tuning phase)
BEST_CONFIG = {
    "cs":      {"model": "sentence_transformers_all_MiniLM_L6_v2", "max_df": 0.8, "min_docs": 200},
    "math":    {"model": "sentence_transformers_all_MiniLM_L6_v2", "max_df": 0.9, "min_docs": 250},
    "physics": {"model": "all_distilroberta_v1",                   "max_df": 0.7, "min_docs": 200},
}

# Create output directories
for subject in LIST_SUBJECT:
    (RESULT_DIR / subject).mkdir(parents=True, exist_ok=True)

print(f"Subjects: {LIST_SUBJECT}")
print(f"Checkpoint directory: {CHECKPOINT_DIR}")
print(f"Results directory: {RESULT_DIR}")

Subjects: ['cs', 'math', 'physics']
Checkpoint directory: ../../../../models/topicGpt
Results directory: ../../../../results/topicGpt/temporal


## Helper Functions

In [3]:
# def load_dataset(subject: str) -> pd.DataFrame:
#     """Load dataset with year column."""
#     file_path = BASE_DIR / subject / "emb" / f"{VERSION}.csv"
#     df = pd.read_csv(file_path)
#     df["submitted_date"] = pd.to_datetime(df["submitted_date"])
#     df["year"] = df["submitted_date"].dt.year
#     return df
def parse_bow_text(text_val):
    """Parse BOW text column: stored as Python list literals."""
    try:
        tokens = ast.literal_eval(text_val)
        if isinstance(tokens, list):
            return tokens
    except (ValueError, SyntaxError):
        pass
    return str(text_val).split()
def load_dataset(subject: str) -> pd.DataFrame:
    """Load dataset with year column. Converts list-literal text to space-separated."""
    file_path = BASE_DIR / subject / "bow" / f"{VERSION}.csv"
    df = pd.read_csv(file_path)
    df["submitted_date"] = pd.to_datetime(df["submitted_date"])
    df["year"] = df["submitted_date"].dt.year
    # Convert list-literal text to space-separated for CountVectorizer
    df["text"] = df["text"].apply(lambda x: " ".join(parse_bow_text(x)))
    return df

def load_tuning_best(subject: str):
    """Load tuning_best.pkl checkpoint."""
    path = CHECKPOINT_DIR / subject / "tuning_best.pkl"
    with open(path, "rb") as f:
        data = pickle.load(f)
    return data


def compute_ctfidf_per_year(
    df, topic_col="topic", text_col="text", top_n=10,
    global_tuning=True, evolution_tuning=True,
):
    """
    Compute c-TF-IDF per topic per year with optional tuning.
    
    Mirrors BERTopic's topics_over_time() tuning approach:
      - global_tuning: average each (year, topic) c-TF-IDF with the
        global (all-year) topic c-TF-IDF to anchor representations.
      - evolution_tuning: average each (year, topic) c-TF-IDF with
        the previous year (t-1) to smooth transitions.
    
    Returns:
        topic_words_per_year: dict of {(year, topic_id): [word1, word2, ...]}
    """
    years = sorted(df["year"].unique())
    topics = sorted(df[topic_col].unique())
    
    # Group documents by (year, topic) and concatenate
    groups = df.groupby(["year", topic_col])[text_col].apply(
        lambda x: " ".join(x)
    ).reset_index()
    groups.columns = ["year", "topic", "text"]
    
    # Build vocabulary across all groups
    vectorizer = CountVectorizer(stop_words="english")
    tf_matrix = vectorizer.fit_transform(groups["text"])
    vocab = vectorizer.get_feature_names_out()
    
    # Compute IDF: log(N / df_t) where N = number of groups, df_t = groups containing term
    n_groups = tf_matrix.shape[0]
    df_t = (tf_matrix > 0).sum(axis=0).A1  # document frequency per term
    idf = np.log((n_groups + 1) / (df_t + 1)) + 1  # smoothed IDF
    
    # TF-IDF per group
    tfidf_matrix = tf_matrix.multiply(idf).toarray()
    
    # --- Global c-TF-IDF (topic only, ignoring year) ---
    global_tfidf = None
    global_topic_to_idx = {}
    if global_tuning:
        global_groups = df.groupby(topic_col)[text_col].apply(
            lambda x: " ".join(x)
        ).reset_index()
        global_groups.columns = ["topic", "text"]
        global_groups = global_groups.sort_values("topic").reset_index(drop=True)
        global_tf = vectorizer.transform(global_groups["text"])
        global_tfidf = global_tf.multiply(idf).toarray()
        # L1 normalise global representation (matches BERTopic)
        global_row_sums = global_tfidf.sum(axis=1, keepdims=True)
        global_row_sums[global_row_sums == 0] = 1
        global_tfidf = global_tfidf / global_row_sums
        global_topic_to_idx = {
            int(t): i for i, t in enumerate(global_groups["topic"])
        }
    
    # --- L1 normalise per-year matrix (before tuning, matches BERTopic) ---
    if global_tuning or evolution_tuning:
        row_sums = tfidf_matrix.sum(axis=1, keepdims=True)
        row_sums[row_sums == 0] = 1
        tfidf_matrix = tfidf_matrix / row_sums
    
    # --- Build index: (year, topic) -> row index ---
    yt_to_idx = {}
    for idx in range(len(groups)):
        y = groups.iloc[idx]["year"]
        t = groups.iloc[idx]["topic"]
        yt_to_idx[(y, t)] = idx
    
    # --- Evolution tuning: average with t-1 ---
    if evolution_tuning:
        for yi in range(1, len(years)):
            curr_year = years[yi]
            prev_year = years[yi - 1]
            for topic in topics:
                curr_key = (curr_year, topic)
                prev_key = (prev_year, topic)
                if curr_key in yt_to_idx and prev_key in yt_to_idx:
                    ci = yt_to_idx[curr_key]
                    pi = yt_to_idx[prev_key]
                    tfidf_matrix[ci] = (tfidf_matrix[ci] + tfidf_matrix[pi]) / 2.0
    
    # --- Global tuning: average with global representation ---
    if global_tuning:
        for idx in range(len(groups)):
            topic = int(groups.iloc[idx]["topic"])
            if topic in global_topic_to_idx:
                gi = global_topic_to_idx[topic]
                tfidf_matrix[idx] = (tfidf_matrix[idx] + global_tfidf[gi]) / 2.0
    
    # --- Final L2 normalise before extracting top words ---
    row_norms = np.linalg.norm(tfidf_matrix, axis=1, keepdims=True)
    row_norms[row_norms == 0] = 1
    tfidf_matrix = tfidf_matrix / row_norms
    
    # Extract top words per (year, topic)
    topic_words_per_year = {}
    for idx in range(len(groups)):
        year = groups.iloc[idx]["year"]
        topic = groups.iloc[idx]["topic"]
        scores = tfidf_matrix[idx]
        top_indices = scores.argsort()[-top_n:][::-1]
        top_words = [vocab[i] for i in top_indices if scores[i] > 0]
        topic_words_per_year[(year, topic)] = top_words
    
    return topic_words_per_year


def calculate_coherence_for_words(
    topic_word_lists,
    texts_tokenized,
    dictionary,
) -> float:
    """Calculate C_v coherence given a list of topic word lists."""
    if len(topic_word_lists) == 0:
        return 0.0
    # Filter out empty or too-short topic word lists
    valid_topics = [tw for tw in topic_word_lists if len(tw) >= 2]
    if len(valid_topics) == 0:
        return 0.0
    
    cm = CoherenceModel(
        topics=valid_topics,
        texts=texts_tokenized,
        dictionary=dictionary,
        coherence='c_v',
        processes=1
    )
    return cm.get_coherence()


def rbo(list1, list2, p=0.9):
    if not list1 and not list2:
        return 1.0
    if not list1 or not list2:
        return 0.0

    # assign short (S) and long (L)
    if len(list1) <= len(list2):
        S, L = list1, list2
    else:
        S, L = list2, list1

    s, l = len(S), len(L)

    S_seen = set()
    L_seen = set()

    X = 0  # overlap
    rbo = 0.0
    disjoint = 0.0
    ext_term = 0.0

    for d in range(l):
        if d < s:
            s_item = S[d]
            S_seen.add(s_item)
        else:
            s_item = None

        l_item = L[d]
        L_seen.add(l_item)

        overlap_incr = 0

        if d < s:
            if s_item == l_item:
                overlap_incr = 1
            else:
                if s_item in L_seen:
                    overlap_incr += 1
                if l_item in S_seen:
                    overlap_incr += 1
        else:
            if l_item in S_seen:
                overlap_incr = 1

        X += overlap_incr

        if d < s:
            A_d = 2.0 * X / (len(S_seen) + len(L_seen))
        else:
            A_d = X / (d + 1)

        rbo += (1 - p) * (p ** d) * A_d

        if d < s:
            ext_term = A_d * (p ** (d + 1))
        else:
            X_s = X - overlap_incr if d == s else X_s
            disjoint += (1 - p) * (p ** d) * (
                X_s * (d + 1 - s) / ((d + 1) * s)
            )
            ext_term = (
                ((X - X_s) / (d + 1) + X_s / s)
                * (p ** (d + 1))
            )

        # optional optimization (safe)
        if p ** d < 1e-12:
            break

    return min(max(rbo + disjoint + ext_term, 0.0), 1.0)

def calculate_irbo(topics_words, p=0.9):
    """Calculate mean IRBO diversity across all topic pairs."""
    if len(topics_words) < 2:
        return 0.0
    irbo_scores = []
    for (i, j) in combinations(range(len(topics_words)), 2):
        similarity = rbo(topics_words[i], topics_words[j], p=p)
        irbo_scores.append(1.0 - similarity)
    return np.mean(irbo_scores)

## Load Assignments & Data

Load the best tuned assignment checkpoint for each subject
and join with the dataset to get `year` from `submitted_date`.

In [4]:
all_data = {}
all_years = {}
all_n_topics = {}

for subject in LIST_SUBJECT:
    print(f"\nLoading {subject}...")
    
    # Load dataset with year
    df_full = load_dataset(subject)
    
    # Load tuning best checkpoint
    ckpt = load_tuning_best(subject)
    df_assign = ckpt["assignment_df"]
    config = ckpt["config"]
    metrics = ckpt["metrics"]
    
    # Join: add year and text to assignments via doc_idx
    df_assign = df_assign.copy()
    df_assign["year"] = df_assign["doc_idx"].map(df_full["year"])
    df_assign["text"] = df_assign["doc_idx"].map(df_full["text"].fillna(""))
    df_assign["topic"] = df_assign["topic_id"]  # alias for consistency
    
    all_data[subject] = df_assign
    years = sorted(df_assign["year"].dropna().unique().astype(int))
    all_years[subject] = years
    n_topics = df_assign["topic_id"].nunique()
    all_n_topics[subject] = n_topics
    
    print(f"  {subject}: {len(df_assign):,} docs, {n_topics} topics, "
          f"{len(years)} years ({years[0]}-{years[-1]})")
    print(f"  Config: model={config['best_model']}, max_df={config['max_df']}, min_docs={config['min_docs']}")
    print(f"  Metrics: C_v={metrics['coherence']:.4f}  IRBO={metrics['irbo']:.4f}  TQ={metrics['topic_quality']:.4f}")

print(f"\n✅ All subjects loaded")


Loading cs...
  cs: 65,516 docs, 138 topics, 26 years (2000-2025)
  Config: model=sentence_transformers_all_MiniLM_L6_v2, max_df=0.8, min_docs=200
  Metrics: C_v=0.7095  IRBO=0.9893  TQ=0.8264

Loading math...
  math: 39,176 docs, 90 topics, 26 years (2000-2025)
  Config: model=sentence_transformers_all_MiniLM_L6_v2, max_df=0.9, min_docs=250
  Metrics: C_v=0.7280  IRBO=0.9809  TQ=0.8358

Loading physics...
  physics: 28,507 docs, 74 topics, 26 years (2000-2025)
  Config: model=all_distilroberta_v1, max_df=0.7, min_docs=200
  Metrics: C_v=0.7398  IRBO=0.9873  TQ=0.8458

✅ All subjects loaded


## Topic Prevalence Over Time

For each year, compute the proportion of documents belonging to each topic.
This shows how topics rise, fall, emerge, and disappear over time.

In [5]:
for subject in LIST_SUBJECT:
    df = all_data[subject]
    years = all_years[subject]
    n_topics = all_n_topics[subject]
    topic_ids = sorted(df["topic"].unique())

    prevalence_csv = RESULT_DIR / subject / "topic_prevalence.csv"

    print(f"\n{'='*70}")
    print(f"Topic Prevalence: {subject.upper()} ({n_topics} topics)")
    print(f"{'='*70}")

    prevalence_rows = []

    for year in years:
        year_df = df[df["year"] == year]
        n_docs_year = len(year_df)
        topic_counts = year_df["topic"].value_counts()

        for topic_id in topic_ids:
            count = topic_counts.get(topic_id, 0)
            proportion = count / n_docs_year if n_docs_year > 0 else 0.0

            # Get topic label from the assignment df
            label_row = df[df["topic"] == topic_id]
            topic_label = label_row["topic_label"].iloc[0] if len(label_row) > 0 else ""
            # Truncate label for display
            short_label = topic_label[:50] if isinstance(topic_label, str) else ""

            prevalence_rows.append({
                "subject": subject,
                "year": year,
                "topic_id": topic_id,
                "topic_label": topic_label,
                "doc_count": count,
                "total_docs_year": n_docs_year,
                "proportion": round(proportion, 6),
            })

        # Summary for this year
        active_topics = (topic_counts > 0).sum()
        top_topic = topic_counts.idxmax()
        top_count = topic_counts.max()
        print(f"  {year}: {n_docs_year:,} docs, {active_topics}/{n_topics} active topics, "
              f"top=T{top_topic} ({top_count} docs)")

    prevalence_df = pd.DataFrame(prevalence_rows)
    prevalence_df.to_csv(prevalence_csv, index=False)
    print(f"\n  Saved to: {prevalence_csv} ({len(prevalence_df)} rows)")


Topic Prevalence: CS (138 topics)
  2000: 60 docs, 25/138 active topics, top=T10 (9 docs)
  2001: 74 docs, 35/138 active topics, top=T10 (6 docs)
  2002: 88 docs, 35/138 active topics, top=T10 (8 docs)
  2003: 100 docs, 31/138 active topics, top=T13 (10 docs)
  2004: 107 docs, 35/138 active topics, top=T29 (13 docs)
  2005: 167 docs, 39/138 active topics, top=T66 (29 docs)
  2006: 174 docs, 38/138 active topics, top=T66 (23 docs)
  2007: 177 docs, 41/138 active topics, top=T6 (26 docs)
  2008: 207 docs, 45/138 active topics, top=T35 (26 docs)
  2009: 171 docs, 40/138 active topics, top=T6 (20 docs)
  2010: 219 docs, 51/138 active topics, top=T6 (18 docs)
  2011: 300 docs, 62/138 active topics, top=T6 (32 docs)
  2012: 406 docs, 75/138 active topics, top=T66 (28 docs)
  2013: 447 docs, 92/138 active topics, top=T6 (27 docs)
  2014: 559 docs, 96/138 active topics, top=T48 (32 docs)
  2015: 831 docs, 116/138 active topics, top=T6 (35 docs)
  2016: 1,152 docs, 127/138 active topics, top=T

## Topic Word Evolution (c-TF-IDF per Year)

Compute c-TF-IDF for each topic at each time point to see how
topic word compositions change over time. This is the core of
BERTopic's `topics_over_time()` approach.

In [6]:
all_topic_words_per_year = {}

for subject in LIST_SUBJECT:
    df = all_data[subject]
    n_topics = all_n_topics[subject]

    evolution_csv = RESULT_DIR / subject / "topic_word_evolution.csv"

    print(f"\n{'='*70}")
    print(f"Topic Word Evolution: {subject.upper()}")
    print(f"{'='*70}")

    start = time.time()
    topic_words_per_year = compute_ctfidf_per_year(
        df, topic_col="topic", text_col="text", top_n=TOP_N_WORDS,
        global_tuning=False, evolution_tuning=True,
    )
    elapsed = time.time() - start
    all_topic_words_per_year[subject] = topic_words_per_year

    print(f"  c-TF-IDF computed in {elapsed:.1f}s")
    print(f"  (year, topic) groups: {len(topic_words_per_year)}")

    # Save word evolution
    evolution_rows = []
    for (year, topic_id), words in sorted(topic_words_per_year.items()):
        evolution_rows.append({
            "subject": subject,
            "year": year,
            "topic_id": topic_id,
            "top_words": ", ".join(words),
        })

    evolution_df = pd.DataFrame(evolution_rows)
    evolution_df.to_csv(evolution_csv, index=False)
    print(f"  Saved to: {evolution_csv}")

    # Show example: topic 0 across a few years
    print(f"\n  Example — Topic 0 word evolution:")
    for year in [2000, 2005, 2010, 2015, 2020, 2025]:
        key = (year, 0)
        if key in topic_words_per_year:
            words = ", ".join(topic_words_per_year[key][:5])
            print(f"    {year}: {words}")


Topic Word Evolution: CS
  c-TF-IDF computed in 6.6s
  (year, topic) groups: 2221
  Saved to: ../../../../results/topicGpt/temporal/cs/topic_word_evolution.csv

  Example — Topic 0 word evolution:
    2000: web, apweb, talkmine, mining, text
    2005: document, keyphrase, web, concept, treillis
    2010: document, wikipedia, text, xml, subcorpora
    2015: document, web, search, relatedness, query
    2020: query, document, search, retrieval, web
    2025: query, document, search, search_engine, text

Topic Word Evolution: MATH
  c-TF-IDF computed in 3.4s
  (year, topic) groups: 2230
  Saved to: ../../../../results/topicGpt/temporal/math/topic_word_evolution.csv

  Example — Topic 0 word evolution:
    2000: bump, index, rolle, szlenk_index, banach_space
    2005: banach_space, tsirelson, separable, space, norm
    2010: banach_space, operator, space, closable, index
    2015: banach_space, operator, space, separable, schauder
    2020: banach_space, operator, space, norm, operatornam

## Per-Year Coherence & IRBO

For each year, compute coherence and IRBO using that year's c-TF-IDF
topic word lists. This measures how well-defined and diverse the topics
are at each point in time.

In [7]:
for subject in LIST_SUBJECT:
    df = all_data[subject]
    years = all_years[subject]
    n_topics = all_n_topics[subject]
    topic_words_per_year = all_topic_words_per_year[subject]

    metrics_csv = RESULT_DIR / subject / "per_year_metrics.csv"

    print(f"\n{"="*70}")
    print(f"Per-Year Metrics: {subject.upper()} ({n_topics} topics)")
    print(f"{"="*70}")

    # Build corpus-level tokenized texts and dictionary (full corpus as reference)
    corpus_texts_tokenized = [text.split() for text in df["text"].tolist()]
    corpus_dictionary = Dictionary(corpus_texts_tokenized)
    print(f"  Corpus: {len(corpus_texts_tokenized):,} docs, {len(corpus_dictionary):,} vocab")

    metrics_rows = []

    for year in years:
        year_df = df[df["year"] == year]
        n_docs = len(year_df)

        active_topics = sorted(year_df["topic"].unique())
        year_topic_words = []
        for tid in active_topics:
            key = (year, tid)
            if key in topic_words_per_year and len(topic_words_per_year[key]) >= 2:
                year_topic_words.append(topic_words_per_year[key])

        coherence = calculate_coherence_for_words(
            year_topic_words, corpus_texts_tokenized, corpus_dictionary
        )
        irbo_mean = calculate_irbo(year_topic_words, p=RBO_P)

        if coherence + irbo_mean > 0:
            topic_quality = 2 * coherence * irbo_mean / (coherence + irbo_mean)
        else:
            topic_quality = 0.0

        n_active = len(active_topics)
        print(f"  {year}: {n_docs:,} docs, {n_active} active | "
              f"Quality={topic_quality:.4f} (C={coherence:.4f}, IRBO={irbo_mean:.4f})")

        metrics_rows.append({
            "subject": subject,
            "year": year,
            "num_docs": n_docs,
            "num_topics_total": n_topics,
            "num_topics_active": n_active,
            "coherence_cv": round(coherence, 6),
            "irbo_mean": round(irbo_mean, 6),
            "topic_quality": round(topic_quality, 6),
        })

    metrics_df = pd.DataFrame(metrics_rows)
    metrics_df.to_csv(metrics_csv, index=False)
    print(f"\n  Saved: {metrics_csv}")


Per-Year Metrics: CS (138 topics)
  Corpus: 65,516 docs, 79,442 vocab
  2000: 60 docs, 25 active | Quality=0.7463 (C=0.5983, IRBO=0.9917)
  2001: 74 docs, 35 active | Quality=0.7043 (C=0.5461, IRBO=0.9915)
  2002: 88 docs, 35 active | Quality=0.6780 (C=0.5152, IRBO=0.9911)
  2003: 100 docs, 31 active | Quality=0.6865 (C=0.5250, IRBO=0.9914)
  2004: 107 docs, 35 active | Quality=0.6527 (C=0.4870, IRBO=0.9894)
  2005: 167 docs, 39 active | Quality=0.7058 (C=0.5479, IRBO=0.9913)
  2006: 174 docs, 38 active | Quality=0.6908 (C=0.5310, IRBO=0.9880)
  2007: 177 docs, 41 active | Quality=0.7258 (C=0.5729, IRBO=0.9902)
  2008: 207 docs, 45 active | Quality=0.7318 (C=0.5808, IRBO=0.9890)
  2009: 171 docs, 40 active | Quality=0.7058 (C=0.5490, IRBO=0.9882)
  2010: 219 docs, 51 active | Quality=0.7253 (C=0.5727, IRBO=0.9886)
  2011: 300 docs, 62 active | Quality=0.6997 (C=0.5414, IRBO=0.9889)
  2012: 406 docs, 75 active | Quality=0.7169 (C=0.5616, IRBO=0.9909)
  2013: 447 docs, 92 active | Quali

## Topic Trends: Emerging, Growing, and Declining Topics

Identify which topics are trending up, trending down,
or stable over the full time period.

In [8]:
from scipy.stats import linregress

for subject in LIST_SUBJECT:
    df = all_data[subject]
    # Get topic words from evolution CSV
    evo_path = RESULT_DIR / subject / "topic_word_evolution.csv"
    evo_df = pd.read_csv(evo_path)
    last_year = evo_df['year'].max()
    last_evo = evo_df[evo_df['year'] == last_year]
    global_tw = {}
    for _, erow in last_evo.iterrows():
        global_tw[int(erow['topic_id'])] = [w.strip() for w in str(erow['top_words']).split(',')][:5]

    rows = []
    topic_ids = sorted(df["topic"].unique())
    for tid in topic_ids:
        topic_df = df[df["topic"] == tid]
        if len(topic_df) == 0:
            continue

        year_counts = topic_df["year"].value_counts().sort_index()
        total_per_year = df["year"].value_counts().sort_index()
        proportions = (year_counts / total_per_year).fillna(0)

        # Align proportions with all years
        all_yrs = sorted(total_per_year.index)
        prop_aligned = proportions.reindex(all_yrs, fill_value=0.0)

        years_arr = np.array(all_yrs, dtype=float)
        props_arr = prop_aligned.values.astype(float)

        # Linear regression
        slope, intercept, r_val, p_val, std_err = linregress(years_arr, props_arr)

        # Early/late for display
        topic_years = sorted(year_counts.index)
        early_mean = proportions[topic_years[:5]].mean() if len(topic_years) >= 5 else proportions.mean()
        late_mean = proportions[topic_years[-5:]].mean() if len(topic_years) >= 5 else proportions.mean()

        # Classify by slope significance
        if p_val < 0.05 and slope > 0:
            trend_label = "GROWING"
        elif p_val < 0.05 and slope < 0:
            trend_label = "DECLINING"
        else:
            trend_label = "STABLE"

        # Get topic label
        label_row = df[df["topic"] == tid]
        topic_label = label_row["topic_label"].iloc[0] if len(label_row) > 0 else ""

        top_words = global_tw.get(tid, ["?"])
        rows.append({
            "subject": subject, "topic_id": tid,
            "topic_label": topic_label,
            "top_words": ", ".join(top_words),
            "total_docs": len(topic_df),
            "first_year": year_counts.index.min(),
            "last_year": year_counts.index.max(),
            "early_proportion": round(early_mean, 6),
            "late_proportion": round(late_mean, 6),
            "slope": round(slope, 8),
            "r_squared": round(r_val**2, 4),
            "p_value": round(p_val, 6),
            "trend": trend_label,
        })

    trends_df = pd.DataFrame(rows)
    trends_df.to_csv(RESULT_DIR / subject / "topic_trends.csv", index=False)

    g = len(trends_df[trends_df["trend"] == "GROWING"])
    s = len(trends_df[trends_df["trend"] == "STABLE"])
    d = len(trends_df[trends_df["trend"] == "DECLINING"])
    print(f"  {subject.upper()}: Growing={g}, Stable={s}, Declining={d}")

  CS: Growing=87, Stable=30, Declining=21
  MATH: Growing=26, Stable=32, Declining=32
  PHYSICS: Growing=24, Stable=32, Declining=18


## Top 5 Growing & Declining Topics

In [9]:
for subject in LIST_SUBJECT:
    trends_df = pd.read_csv(RESULT_DIR / subject / "topic_trends.csv")

    print(f"\n{'='*80}")
    print(f"  {subject.upper()}")
    print(f"{'='*80}")

    growing = trends_df[trends_df['trend'] == 'GROWING'].sort_values('slope', ascending=False)
    declining = trends_df[trends_df['trend'] == 'DECLINING'].sort_values('slope', ascending=True)

    print(f"\n  " + chr(0x1F4C8) + f" TOP 5 GROWING (steepest positive slope):")
    for _, row in growing.head(5).iterrows():
        print(f"    T{int(row['topic_id']):>3} | slope={row['slope']:+.6f} "
              f"R\u00B2={row['r_squared']:.3f} | "
              f"{row['early_proportion']:.4f} \u2192 {row['late_proportion']:.4f} | "
              f"{row['top_words']}")

    print(f"\n  " + chr(0x1F4C9) + f" TOP 5 DECLINING (steepest negative slope):")
    for _, row in declining.head(5).iterrows():
        print(f"    T{int(row['topic_id']):>3} | slope={row['slope']:+.6f} "
              f"R\u00B2={row['r_squared']:.3f} | "
              f"{row['early_proportion']:.4f} \u2192 {row['late_proportion']:.4f} | "
              f"{row['top_words']}")


  CS

  📈 TOP 5 GROWING (steepest positive slope):
    T 72 | slope=+0.001826 R²=0.687 | 0.0181 → 0.0338 | segmentation, image, medical, dataset, datum
    T119 | slope=+0.001359 R²=0.728 | 0.0050 → 0.0307 | video, image, attention, diffusion, transformer
    T110 | slope=+0.001244 R²=0.667 | 0.0058 → 0.0285 | llm, language, training, datum, dataset
    T121 | slope=+0.001241 R²=0.739 | 0.0037 → 0.0268 | language, llm, multilingual, datum, fine_tuning
    T100 | slope=+0.001117 R²=0.652 | 0.0115 → 0.0217 | adversarial, attack, image, adversarial_attack, robustness

  📉 TOP 5 DECLINING (steepest negative slope):
    T 10 | slope=-0.003644 R²=0.648 | 0.0951 → 0.0072 | program, llm, language, code, specification
    T  0 | slope=-0.003192 R²=0.674 | 0.0769 → 0.0035 | query, document, search, search_engine, text
    T 34 | slope=-0.002838 R²=0.777 | 0.0572 → 0.0022 | polynomial, csp, formula, circuit, boolean
    T 66 | slope=-0.002650 R²=0.182 | 0.0755 → 0.0029 | channel, code, rate, exp

## Evolution Summary

In [10]:
for subject in LIST_SUBJECT:
    metrics_csv = RESULT_DIR / subject / "per_year_metrics.csv"
    trends_csv = RESULT_DIR / subject / "topic_trends.csv"

    if not metrics_csv.exists():
        print(f"{subject}: missing metrics, skipping")
        continue

    metrics_df = pd.read_csv(metrics_csv)
    trends_df = pd.read_csv(trends_csv)

    n_topics = all_n_topics[subject]

    growing = len(trends_df[trends_df["trend"] == "GROWING"])
    declining = len(trends_df[trends_df["trend"] == "DECLINING"])
    stable = len(trends_df[trends_df["trend"] == "STABLE"])

    summary_data = {
        "subject": subject,
        "num_topics": n_topics,
        "num_years": len(metrics_df),
        "coherence_mean": round(metrics_df["coherence_cv"].mean(), 6),
        "coherence_std": round(metrics_df["coherence_cv"].std(), 6),
        "irbo_mean": round(metrics_df["irbo_mean"].mean(), 6),
        "irbo_std": round(metrics_df["irbo_mean"].std(), 6),
        "quality_mean": round(metrics_df["topic_quality"].mean(), 6),
        "quality_std": round(metrics_df["topic_quality"].std(), 6),
        "topics_growing": growing,
        "topics_stable": stable,
        "topics_declining": declining,
    }
    summary_df = pd.DataFrame([summary_data])
    summary_csv = RESULT_DIR / subject / "evolution_summary.csv"
    summary_df.to_csv(summary_csv, index=False)
    print(f"\n  {subject.upper()} summary saved to: {summary_csv}")


  CS summary saved to: ../../../../results/topicGpt/temporal/cs/evolution_summary.csv

  MATH summary saved to: ../../../../results/topicGpt/temporal/math/evolution_summary.csv

  PHYSICS summary saved to: ../../../../results/topicGpt/temporal/physics/evolution_summary.csv


## Final Results

In [11]:
print("\n" + "=" * 110)
print("TOPICGPT TEMPORAL ANALYSIS: FINAL RESULTS")
print("=" * 110)

for subject in LIST_SUBJECT:
    metrics_csv = RESULT_DIR / subject / "per_year_metrics.csv"
    trends_csv = RESULT_DIR / subject / "topic_trends.csv"

    if not metrics_csv.exists():
        print(f"\n{subject.upper()}: No results found")
        continue

    metrics_df = pd.read_csv(metrics_csv)
    trends_df = pd.read_csv(trends_csv)

    n_topics = all_n_topics[subject]

    growing = len(trends_df[trends_df["trend"] == "GROWING"])
    declining = len(trends_df[trends_df["trend"] == "DECLINING"])
    stable = len(trends_df[trends_df["trend"] == "STABLE"])

    config = BEST_CONFIG[subject]

    sep = chr(9472)
    print(f"\n{sep*60}")
    print(f"  Subject:        {subject.upper()}")
    print(f"  Num topics:     {n_topics}")
    print(f"  Model:          {config['model']}")
    print(f"  max_df:         {config['max_df']}")
    print(f"  min_docs:       {config['min_docs']}")
    print(f"  Years:          {len(metrics_df)}")
    print(f"  Coherence:      {metrics_df['coherence_cv'].mean():.4f} +/- {metrics_df['coherence_cv'].std():.4f}")
    print(f"  IRBO:           {metrics_df['irbo_mean'].mean():.4f} +/- {metrics_df['irbo_mean'].std():.4f}")
    print(f"  Topic Quality:  {metrics_df['topic_quality'].mean():.4f} +/- {metrics_df['topic_quality'].std():.4f}")
    print(f"  Trends:         ↑{growing} growing, →{stable} stable, ↓{declining} declining")
    print(f"{sep*60}")

print("\n" + "=" * 110)
print("Per-Year Details:")
print("=" * 110)

for subject in LIST_SUBJECT:
    metrics_csv = RESULT_DIR / subject / "per_year_metrics.csv"
    if not metrics_csv.exists():
        continue

    metrics_df = pd.read_csv(metrics_csv)

    print(f"\n{subject.upper()}:")
    print(metrics_df[["year", "num_docs", "num_topics_active",
                     "coherence_cv", "irbo_mean", "topic_quality"]].to_string(index=False))
    print()


TOPICGPT TEMPORAL ANALYSIS: FINAL RESULTS

────────────────────────────────────────────────────────────
  Subject:        CS
  Num topics:     138
  Model:          sentence_transformers_all_MiniLM_L6_v2
  max_df:         0.8
  min_docs:       200
  Years:          26
  Coherence:      0.5766 +/- 0.0549
  IRBO:           0.9816 +/- 0.0123
  Topic Quality:  0.7246 +/- 0.0398
  Trends:         ↑87 growing, →30 stable, ↓21 declining
────────────────────────────────────────────────────────────

────────────────────────────────────────────────────────────
  Subject:        MATH
  Num topics:     90
  Model:          sentence_transformers_all_MiniLM_L6_v2
  max_df:         0.9
  min_docs:       250
  Years:          26
  Coherence:      0.5810 +/- 0.0753
  IRBO:           0.9779 +/- 0.0059
  Topic Quality:  0.7259 +/- 0.0587
  Trends:         ↑26 growing, →32 stable, ↓32 declining
────────────────────────────────────────────────────────────

─────────────────────────────────────────────────